# CASTalign Testground

Two-hop alignment pipeline: BARseq ex-vivo slices → ex-vivo block → in-vivo stack.

## Pipeline Structure
```
slice_10 ──→ ex_vivo_block_red ──→ invivo_ref_red
slice_22 ──→ ex_vivo_block_red ──→ invivo_ref_red
...
```

## Workflow
1. **Load graph** with in-vivo reference, ex-vivo block, and subslices
2. **Select slice** via dropdown widget
3. **Align slices → block** using one of three modes:
   - Mode A: Padded, block as base (point-based, RAM-heavy)
   - Mode B: No padding, block as base (parametric only, low RAM)
   - Mode C: No padding, slice as base / block movable (point-based, low RAM)
4. **Align block → in-vivo** (Mode D, done once — 3D vs 3D)
5. **Verify** alignment with Pearson correlation + napari overlay
6. **Refine** with Triangulation2D for edge warping
7. **Export** warped images as TIFF
8. **Visualize** all aligned slices with GraphViewer

---

## 1. Imports & Setup

In [ ]:
# =============================================================================
# IMPORTS & SETUP
# =============================================================================
%reset -f

import sys
import gc
from pathlib import Path
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
from scipy.stats import pearsonr

# =============================================================================
# PATH SETUP
# =============================================================================
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent  # cell2cell-alignment/

# Add project root to path for local_config and utilities
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# =============================================================================
# LOCAL CONFIG (user-specific paths)
# =============================================================================
try:
    from local_config import GRAPH_PATH
except ImportError:
    raise ImportError(
        "local_config.py not found.\n"
        "Copy local_config.example.py to local_config.py and fill in your paths:\n"
        "    cp local_config.example.py local_config.py"
    )

# =============================================================================
# CASTALIGN IMPORTS
# =============================================================================
try:
    import castalign as ca
    import castalign.gui as ca_gui
    from castalign.gui import GraphViewer
    from castalign import ndarray_shifted
    import castalign.utils as ca_utils
    print("Imported castalign")
except ImportError:
    # Fallback to linestuffup if castalign not available
    import linestuffup as ca
    import linestuffup.gui as ca_gui
    from linestuffup import ndarray_shifted
    import linestuffup.utils as ca_utils
    try:
        from linestuffup.gui import GraphViewer
    except ImportError:
        GraphViewer = None
        print("Warning: GraphViewer not available in this version")
    print("Imported linestuffup (castalign fallback)")

# =============================================================================
# PROJECT UTILITIES
# =============================================================================
from utilities.graph_viz_fix import patch_castalign_visualise
patch_castalign_visualise()

# =============================================================================
# JUPYTER MAGIC
# =============================================================================
%load_ext autoreload
%autoreload 2

# Disable label auto-detection
ca_utils.image_is_label = lambda img: False

print()
print("="*60)
print("CASTALIGN TESTGROUND")
print("="*60)
print("Imports complete")
print("GraphViz patched (SVG output)")
print("Auto-reload enabled")
print("="*60)

## 2. Path Configuration

In [ ]:
# =============================================================================
# PATH CONFIGURATION (from local_config.py)
# =============================================================================
#
# All paths are set in local_config.py (project root, gitignored).
# Edit that file to change data locations. It contains:
#   DATA_ROOT        — raw BARseq data
#   OUTPUT_ROOT      — all pipeline output
#   BLOCK_STACK_PATH — ex-vivo block TIFF
#   GRAPH_PATH       — alignment graph .db file
#
# Template: local_config.example.py
#
# WARNING: SQLite over Samba (Mac accessing server files)
# --------------------------------------------------------
# The graph .db files are SQLite databases. Writing SQLite over network mounts
# (Samba/SMB) can sometimes cause issues:
#   - "database is locked" errors
#   - Corruption errors on save
#   - Hanging during graph.save()
#
# If you encounter these issues, consider:
#   1. Copy graph locally, align, copy back
#   2. Run alignment directly on the server (SSH + X11 forwarding)
# =============================================================================

GRAPH_PATH = Path(GRAPH_PATH)
WARPED_OUTPUT_DIR = GRAPH_PATH.parent / "warped_output"

print("Path verification:")
print(f"  Graph file:    {'OK' if GRAPH_PATH.exists() else 'NOT FOUND'} {GRAPH_PATH}")
print(f"  Warped output: {'OK' if WARPED_OUTPUT_DIR.exists() else 'WILL BE CREATED'} {WARPED_OUTPUT_DIR}")

## 3. Load Graph

In [ ]:
# =============================================================================
# LOAD GRAPH
# =============================================================================
if not GRAPH_PATH.exists():
    raise FileNotFoundError(f"Graph not found: {GRAPH_PATH}
Run subslice_graph_builder.py to build it first.")

g = ca.Graph.load(str(GRAPH_PATH))

# Node names - the alignment-canonical (red) channel of each 2P volume.
# Sibling green nodes (invivo_ref_green / ex_vivo_block_green) are joined to
# these via castalign.base.Identity edges and compose for free.
invivo_node = "invivo_ref_red"
block_node = "ex_vivo_block_red"

# Non-slice node names: both 2P-volume channels and the `ex_vivo_rigid` ref
# node (added post-promote in cell 9b). Subtracted from `slice_nodes` below.
NON_SLICE_NODES = {
    "invivo_ref_red", "invivo_ref_green",
    "ex_vivo_block_red", "ex_vivo_block_green",
    "ex_vivo_rigid",
}

# =============================================================================
# VERIFY EXPECTED NODES
# =============================================================================
all_nodes = list(g.nodes)

if block_node not in g.nodes:
    raise ValueError(f"'{block_node}' not in graph. Rebuild graph with block_path set.")

slice_nodes = sorted([n for n in all_nodes if n not in NON_SLICE_NODES])

aligned_slices = []
unaligned_slices = []
for s in slice_nodes:
    if s in g.edges and block_node in g.edges[s]:
        aligned_slices.append(s)
    else:
        unaligned_slices.append(s)

block_to_invivo_aligned = (
    block_node in g.edges and invivo_node in g.edges.get(block_node, {})
)

print("="*60)
print("GRAPH LOADED")
print("="*60)
print(f"Graph: {g.name}")
print(f"Path:  {GRAPH_PATH}")
print(f"Total nodes: {len(all_nodes)}")
print(f"Subslices: {len(slice_nodes)}")
print(f"  Aligned to block: {len(aligned_slices)}")
print(f"  Unaligned: {len(unaligned_slices)}")
print(f"Block -> InVivo: {'ALIGNED' if block_to_invivo_aligned else 'NOT ALIGNED (run Mode D)'}")
print("="*60)


## 4. Slice Selector

Use the dropdown to select which slice to align. Green = aligned, Red = unaligned.

In [ ]:
# =============================================================================
# SLICE SELECTOR WIDGET
# =============================================================================
import re

def extract_slice_number(node_name):
    """Extract slice number from node name like 'slice10_subslice_mScarlet_cellmask'."""
    match = re.match(r'slice(\d+)', node_name)
    return int(match.group(1)) if match else 0

# Sort slice nodes by number
slice_nodes = sorted(slice_nodes, key=extract_slice_number)

# Create options with alignment status
def get_slice_options():
    """Generate dropdown options with alignment status and transform type."""
    options = []
    for s in slice_nodes:
        is_aligned = s in aligned_slices
        num = extract_slice_number(s)
        if is_aligned:
            t = g.get_transform(s, block_node)
            tname = type(t).__name__
            options.append((f"OK [{tname}] Slice {num}", s))
        else:
            options.append((f"o  Slice {num}", s))
    return options

# Create widget
slice_dropdown = widgets.Dropdown(
    options=get_slice_options(),
    description="Select:",
    style={"description_width": "60px"},
    layout=widgets.Layout(width="350px")
)

# Info display
info_output = widgets.Output()

def show_slice_info(node):
    """Display info for a slice node."""
    with info_output:
        clear_output(wait=True)
        img = g.get_image(node)
        is_aligned = node in aligned_slices
        
        print(f"Selected: {node}")
        print(f"Shape: {img.shape}")
        print(f"Status: {'Aligned to block' if is_aligned else 'Not aligned'}")
        
        if is_aligned:
            t = g.get_transform(node, block_node)
            print(f"Transform: {type(t).__name__}")

def on_slice_change(change):
    """Update info when slice selection changes."""
    show_slice_info(change["new"])

# Attach observer FIRST
slice_dropdown.observe(on_slice_change, names="value")

# Display widget
print("Select a slice to align:")
print("  OK [TransformType] = aligned to block")
print("  o  = not yet aligned")
print()
display(widgets.HBox([slice_dropdown, info_output]))

# Show initial info (observer won't fire since value hasn't changed)
show_slice_info(slice_dropdown.value)

def get_selected_slice():
    """Get currently selected slice name."""
    return slice_dropdown.value

## 5. Alignment Utilities

Helper functions used by all alignment modes.

In [ ]:
# =============================================================================
# ALIGNMENT UTILITIES
# =============================================================================

def get_initial_transform_slice_to_block():
    """
    Initial transform for slice -> ex-vivo block alignment.
    
    Both are ex-vivo tissue, so no large rotation is expected.
    The block is a 3D 2-photon volume of the tissue before slicing.
    
    TODO: May need adjustment (e.g., xrotate=90) depending on how
    the block was imaged relative to the slices. Test and adjust.
    """
    return ca.Identity()


def load_and_pad_slice(slice_name, pad_z=25):
    """
    Load a slice from the graph and pad it in z for point-based transforms.
    
    Parameters
    ----------
    slice_name : str
        Name of the slice node in the graph
    pad_z : int
        Number of blank slices above and below (total z = 2*pad_z + 1)
    
    Returns
    -------
    padded : np.ndarray
        Padded volume with original slice at z=pad_z
    """
    slice_img = g.get_image(slice_name)
    padding_value = slice_img.mean() * 0.90
    padded = np.full((2 * pad_z + 1, *slice_img.shape[1:]), padding_value, dtype=slice_img.dtype)
    padded[pad_z] = slice_img[0]
    return padded


def compute_alignment_quality(fixed_img, movable_img, transform):
    """
    Apply a transform and compute Pearson correlation as alignment quality metric.
    
    Parameters
    ----------
    fixed_img : np.ndarray
        The fixed/reference image
    movable_img : np.ndarray
        The movable image (pre-transform)
    transform : Transform
        The alignment transform (movable -> fixed direction)
    
    Returns
    -------
    r : float
        Pearson correlation coefficient between overlapping regions
    warped : np.ndarray
        The transformed movable image in fixed space
    """
    warped = transform.transform_image(
        movable_img,
        output_size=list(fixed_img.shape),
    )
    warped_arr = np.asarray(warped).astype(np.float64)
    fixed_arr = np.asarray(fixed_img).astype(np.float64)
    
    # Mask to overlapping non-zero region
    mask = (warped_arr > 0) & (fixed_arr > 0)
    if mask.sum() < 10:
        print("Warning: fewer than 10 overlapping non-zero voxels")
        return 0.0, warped_arr
    
    r, _ = pearsonr(fixed_arr[mask], warped_arr[mask])
    return r, warped_arr


def export_warped_image(warped_array, output_path, dtype=None):
    """
    Save a warped volume as a multi-page TIFF.
    
    Parameters
    ----------
    warped_array : np.ndarray
        3D array (Z, Y, X) to save
    output_path : str or Path
        Output file path (.tif or .tiff)
    dtype : numpy dtype, optional
        Cast to this dtype before saving. If None, keeps original.
    """
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    arr = np.asarray(warped_array)
    if dtype is not None:
        arr = arr.astype(dtype)
    
    imageio.mimwrite(str(output_path), [arr[z] for z in range(arr.shape[0])])
    print(f"Saved: {output_path}")
    print(f"  Shape: {arr.shape}, dtype: {arr.dtype}")
    print(f"  Size: {output_path.stat().st_size / 1e6:.1f} MB")


def save_alignment(slice_name, transform):
    """
    Save slice -> block alignment to graph and disk.
    
    Parameters
    ----------
    slice_name : str
        Name of the slice node
    transform : Transform
        The fitted transform (slice -> block direction)
    """
    global aligned_slices, unaligned_slices
    
    # Check if edge already exists to decide update flag
    edge_exists = (slice_name, block_node) in g
    g.add_edge(slice_name, block_node, transform, update=edge_exists)
    
    # Save to disk
    g.save()
    
    # Update tracking lists
    if slice_name not in aligned_slices:
        aligned_slices.append(slice_name)
    if slice_name in unaligned_slices:
        unaligned_slices.remove(slice_name)
    
    # Update dropdown options
    slice_dropdown.options = get_slice_options()
    
    print(f"\nSaved: {slice_name} -> {block_node}")
    print(f"  Transform: {type(transform).__name__}")
    print(f"  Graph saved to: {GRAPH_PATH.name}")


def get_previous_transform(slice_name):
    """
    Get transform from a nearby aligned slice for rotation carry-forward.
    
    Returns the transform from the closest lower-numbered aligned slice,
    or None if no previous slices are aligned.
    """
    current_num = extract_slice_number(slice_name)
    if current_num == 0:
        return None
    
    best_match = None
    best_diff = float("inf")
    
    for s in aligned_slices:
        num = extract_slice_number(s)
        if num > 0 and num < current_num:
            diff = current_num - num
            if diff < best_diff:
                best_diff = diff
                best_match = s
    
    if best_match:
        return g.get_transform(best_match, block_node)
    return None


print("Alignment utilities loaded:")
print("  - get_initial_transform_slice_to_block()")
print("  - load_and_pad_slice(slice_name, pad_z=25)")
print("  - compute_alignment_quality(fixed, movable, transform)")
print("  - export_warped_image(warped, output_path, dtype=None)")
print("  - save_alignment(slice_name, transform)")
print("  - get_previous_transform(slice_name)")

---

## Transform Reference & Geometry Guide

### Transform Keybindings

`align_interactive()` prompts you to choose a transform. These are the available keys:

| Key | Transform Class | Type | Best for |
|-----|----------------|------|----------|
| `t` | TranslateRotateFixed | Parametric (sliders) | Quick slider-based rigid alignment |
| `r` | TranslateRotateRescaleFixed | Parametric (sliders) | Slider-based with scaling |
| `T` | TranslateRotate | Point-based | Precise rigid alignment via landmarks |
| `R` | TranslateRotateRescale | Point-based | Full affine via landmarks (cake) |
| `P` | TranslateRotateRescaleByPlane | Point-based | Pancake/rice-paper with scaling |
| `N` | Triangulation2D | Point-based | Nonlinear refinement for pancake/rice-paper (projected) |
| `V` | Triangulation | Point-based | Nonlinear refinement for cake (3D) |

### Modifier Keys (during GUI)

| Key | Action |
|-----|--------|
| `e` | Edit the previous transform (re-open GUI with current params) |
| `z` | Remove the most recent transform layer |
| `u` | Undo — revert to previous transform state |
| `f` | Flip movable along z axis |
| `d` | Toggle reference images on/off |
| `x_` | Extend previous point-based transform with a new one (e.g., `xN` = add Triangulation2D) |
| `c_` | Convert previous point-based transform type (e.g., `cR` = convert to TranslateRotateRescale) |
| `s` | Save transform to graph (in memory only) |
| `S` | Save transform to graph AND write .db to disk |
| `q` | Quit — return the current transform |
| `v` | View only — opens GUI but doesn't change the transform |

### Geometry Guide

The choice of transform depends on the **relative dimensionality** of your images:

- **Cake** (3D vs 3D, same dimensions): Both volumes have z-extent. Use `R` (affine) then `V` (nonlinear 3D triangulation).
- **Pancake** (3D vs 3D, different z-extent): One volume has less z range. Use `P` (plane-based rescale) then `N` (projected triangulation).
- **Rice paper** (2D slice vs 3D volume): The 2D slice has only 1 z-plane. **Must pad** to enable point-based transforms. Use `T` (rigid) then `N` (projected triangulation).

**Padding rationale:** Point-based transforms need at least a few z-slices to fit 3D correspondences. We pad the 1-voxel slice to `2*PAD_Z+1` slices (default 51) with the original at the center. The padding value is 90% of the slice mean to avoid confusing the transform fitter.

---

## 6. Alignment Mode A: Padded, Block as Base

**Use when:** You need point-based transforms (T, R, N, V, P) on rice-paper slices.

**How it works:** 
- Pads the 1-voxel slice to give it z-extent, enabling point-based fitting
- Ex-vivo block is the base (fixed) image

**RAM usage:** ~400-500 MB

**Transform type:** TranslateRotate (point-based)

**Controls:**
- Click corresponding points on base and movable
- Press 'q' to quit and save

In [ ]:
# =============================================================================
# MODE A: PADDED SLICE (MOVABLE) -> BLOCK (FIXED), point-based, RAM-heavy
# =============================================================================
# Roles: movable = padded 2D BARseq slice, fixed = ex-vivo block (3D).
# Convention: ex-vivo stays fixed; here both sides are ex-vivo but the
# block is the larger/base volume, so it's the fixed one.
# =============================================================================

slice_name = get_selected_slice()
print(f"Aligning: {slice_name}")
print(f"Mode A: padded slice (movable) -> block (fixed)")
print()

# Pad slice (movable) so point-based transforms have z-extent
PAD_Z = 25
movable_img = load_and_pad_slice(slice_name, pad_z=PAD_Z)
fixed_img   = g.get_image(block_node)

print(f"Movable (padded {slice_name}): {movable_img.shape}")
print(f"Fixed   ({block_node}):        {fixed_img.shape}")
print(f"Movable memory: {movable_img.nbytes / 1e6:.1f} MB")

# Initial transform: carry forward from previous slice if available
prev_transform = get_previous_transform(slice_name)
if prev_transform is not None:
    initial_transform = prev_transform
    print(f"\nUsing transform from previous aligned slice")
else:
    initial_transform = get_initial_transform_slice_to_block()
    print(f"\nUsing default initial transform (slice -> block)")

print("\n" + "="*60)
print("LAUNCHING GUI")
print("="*60)
print(f"Movable: padded {slice_name} (green)")
print(f"Fixed:   {block_node} (magenta)")
print("Press 'q' to quit when done")
print()

transform = ca_gui.align_interactive(
    nodes_movable=movable_img,
    nodes_fixed=fixed_img,
    transform=initial_transform,
    graph=g,
)

save_alignment(slice_name, transform)

del movable_img
gc.collect()


---

## 7. Alignment Mode B: No Padding, Block as Base (Parametric)

**Use when:** You only need slider-based alignment, no keypoints.

**How it works:** 
- Uses the raw 1-voxel slice directly (no padding)
- Parametric (slider-based) transforms only

**RAM usage:** Minimal (~50 MB for slice + block)

**Transform type:** TranslateRotateFixed (slider-based)

**Controls:**
- Use sliders to adjust rotation and translation
- Press 'q' to quit and save

**Note:** No point clicking in this mode. For point-based, use Mode A or C.

In [ ]:
# =============================================================================
# MODE B: UNPADDED SLICE (MOVABLE) -> BLOCK (FIXED), parametric only, low RAM
# =============================================================================
# Roles: movable = unpadded 2D slice (1 z-plane), fixed = ex-vivo block.
# No padding -> much lower RAM but point-based transforms (T/R/P/N/V) don't
# fit well without z-extent on the movable. Use parametric (t, r) only.
# =============================================================================

slice_name = get_selected_slice()
print(f"Aligning: {slice_name}")
print(f"Mode B: unpadded slice (movable) -> block (fixed), parametric only")
print()

movable_img = g.get_image(slice_name)
fixed_img   = g.get_image(block_node)

print(f"Movable ({slice_name}): {movable_img.shape}")
print(f"Fixed   ({block_node}): {fixed_img.shape}")
print(f"Movable memory: {movable_img.nbytes / 1e6:.1f} MB (no padding)")

prev_transform = get_previous_transform(slice_name)
if prev_transform is not None:
    initial_transform = prev_transform
    print(f"\nUsing transform from previous aligned slice")
else:
    initial_transform = get_initial_transform_slice_to_block()
    print(f"\nUsing default initial transform (slice -> block)")

print("\n" + "!"*60)
print("NOTE: No padding - point-based transforms (T, R, P) may not work well.")
print("Use parametric transforms (t, r) or switch to Mode A for point-based.")
print("!"*60)

print("\n" + "="*60)
print("LAUNCHING GUI")
print("="*60)
print(f"Movable: {slice_name} (green)")
print(f"Fixed:   {block_node} (magenta)")
print("Press 'q' to quit when done")
print()

transform = ca_gui.align_interactive(
    nodes_movable=movable_img,
    nodes_fixed=fixed_img,
    transform=initial_transform,
    graph=g,
)

save_alignment(slice_name, transform)


---

## 8. Alignment Mode C: Padded 2D as Base (Block Movable)

**Use when:** You want to scroll through the block volume to find where the 2D slice fits.

**How it works:** 
- 2D slice is base with padding (PAD_Z=25, same as Mode A)
- Block volume is movable
- No pre-transforms - manually rotate in napari as needed

**RAM usage:** ~300-400 MB (block ~200MB + padded slice ~100MB)

**Benefits:**
- All transforms work (T, R, N, V, P) due to padding
- Can scroll through block to find matching z-location
- Full manual control over orientation

In [ ]:
# =============================================================================
# MODE C: BLOCK (MOVABLE) -> PADDED SLICE (FIXED), point-based, low RAM
# =============================================================================
# Roles: movable = ex-vivo block (3D), fixed = padded 2D slice.
# Roles are INVERTED vs Modes A/B -- this is the one exception to the
# "ex-vivo fixed" convention. Why: putting the smaller padded slice as
# fixed keeps point-based transforms feasible while cutting the RAM cost
# of rendering the big block as the base. Edge is saved block -> slice
# but castalign stores the inverse automatically (bidirectional).
# =============================================================================

slice_name = get_selected_slice()
print(f"Aligning: {slice_name}")
print(f"Mode C: block (movable) -> padded slice (fixed)")
print()

PAD_Z = 25
fixed_img   = load_and_pad_slice(slice_name, pad_z=PAD_Z)   # padded slice as base
movable_img = g.get_image(block_node)                        # block as movable

print(f"Movable ({block_node}):        {movable_img.shape}")
print(f"Fixed   (padded {slice_name}): {fixed_img.shape}")
print(f"Fixed memory: {fixed_img.nbytes / 1e6:.1f} MB")

# Start from identity (block movable has no per-slice prior transform)
initial_transform = ca.Identity()

print("\n" + "="*60)
print("LAUNCHING GUI")
print("="*60)
print(f"Movable: {block_node} (green)")
print(f"Fixed:   padded {slice_name} (magenta)")
print("Use napari controls to manually rotate/adjust view")
print("Press 'q' to quit when done")
print()

transform_block_to_slice = ca_gui.align_interactive(
    nodes_movable=movable_img,
    nodes_fixed=fixed_img,
    transform=initial_transform,
    graph=g,
)

# Save: edge direction matches movable->fixed (block -> slice)
# Castalign stores the inverse on the reverse edge automatically.
edge_exists = (block_node, slice_name) in g
g.add_edge(block_node, slice_name, transform_block_to_slice, update=edge_exists)
g.save()

# Update tracking
if slice_name not in aligned_slices:
    aligned_slices.append(slice_name)
if slice_name in unaligned_slices:
    unaligned_slices.remove(slice_name)
slice_dropdown.options = get_slice_options()

print(f"\nSaved edge: {block_node} -> {slice_name} (inverse auto-stored)")
print(f"  Transform: {type(transform_block_to_slice).__name__}")
print(f"  Graph saved to: {GRAPH_PATH.name}")

del fixed_img
gc.collect()


---

## 9. Mode D: Block → In-Vivo Alignment (3D vs 3D)

**Do this once.** Aligns the ex-vivo block to the in-vivo reference. Both are 3D 2-photon volumes with the same voxel size.

Once this edge exists, `g.get_transform(slice, invivo_ref_red)` automatically composes the full two-hop path: slice → block → invivo.

**How it works:**
- Both images loaded from the main graph `g`
- Same modality (2P), same resolution — pure spatial registration
- Use `R` (affine) then `V` (3D triangulation) for best results

**RAM usage:** Depends on volume size (~200 MB per volume)

In [ ]:
# =============================================================================
# MODE D: Ex-Vivo Block <-> In-Vivo Alignment (3D vs 3D)
# =============================================================================
# Uses castalign's native node-name mode: pass node names (strings) and let
# castalign resolve images via graph.get_image() and auto-detect the starting
# transform from any existing edge (else Identity).
# =============================================================================
print("="*60)
print("MODE D: Ex-Vivo Block <-> In-Vivo Alignment")
print("="*60)
print()

# -----------------------------------------------------------------
# CONVENTION: in-vivo is MOVABLE, ex-vivo block is FIXED (base).
# Rationale: in-vivo is typically the smaller volume -- cheaper to
# resample each GUI update. Graph edges are bidirectional (castalign
# stores the inverse automatically), so downstream traversal is
# unaffected. To swap roles, just reassign movable_node/fixed_node.
# -----------------------------------------------------------------
movable_node = invivo_node     # in-vivo reference (green in GUI)
fixed_node   = block_node      # ex-vivo block (magenta in GUI, base)

# For reporting only -- castalign loads images itself in node-name mode.
movable_shape = np.asarray(g.get_image(movable_node)).shape
fixed_shape   = np.asarray(g.get_image(fixed_node)).shape
print(f"Movable ({movable_node}): {movable_shape}")
print(f"Fixed   ({fixed_node}): {fixed_shape}")

# Launch alignment GUI
print("\n" + "="*60)
print("LAUNCHING GUI")
print("="*60)
print(f"Movable: {movable_node} (green)")
print(f"Fixed:   {fixed_node} (magenta)")
print("Recommended: R (affine) then V (nonlinear 3D triangulation)")
print("Press 'S' in the GUI to save the edge to disk; 'q' to quit")
print()

# castalign's S (save to disk) / s (save in-memory) / q (quit) hotkeys
# handle all persistence inside align_interactive -- no post-GUI save
# needed (see castalign/gui.py:562-573).
t_movable_to_fixed = ca_gui.align_interactive(
    nodes_movable=movable_node,   # string -> castalign resolves via graph
    nodes_fixed=fixed_node,       # string -> castalign resolves via graph
    graph=g,                      # transform omitted -> auto-detect from graph
)

# Update status
block_to_invivo_aligned = True

print(f"\nGUI closed. Transform returned: {type(t_movable_to_fixed).__name__}")
print(f"  Edge {movable_node} -> {fixed_node} persisted iff you pressed 'S' in the GUI.")
print(f"  Graph file: {GRAPH_PATH.name}")


---

## 9b. Stage 2 — Promote rigid edge to chain

Converts the single rigid edge `invivo_ref_red → ex_vivo_block_red` into the chain
`ex_vivo_block_red → ex_vivo_rigid → invivo_ref_red`, where `ex_vivo_rigid` is a
reference node (zero data duplication — shares image with `ex_vivo_block_red` per
`castalign/graph.py:244-247`). After this, both rigid-only and composed
transforms are independently queryable.

The ref node sits on the **ex-vivo** side so the nonlinear edge's forward
direction ends at in-vivo (the project's ground-truth frame). Forward
traversal of the nonlinear leg is the fast/exact path via
`castalign/base.py:358`, and the 1000-point hard cap on numerical inversion
makes fit direction load-bearing for MNN validation. See
`project_invivo_anatomical_ground_truth.md` and `project_nonlinear_fit_direction.md`
(rule #4: forward = most-distorted → ground truth).

**WARNING:** After promote, do NOT re-run Stage 1 (Mode D) on
`invivo_ref_red → ex_vivo_block_red` — castalign's `S` hotkey would write a direct
1-hop edge that shadows the chain. To re-fit rigid: run Stage 4 (demote) →
Mode D → Stage 2 (promote) again.


In [ ]:
# =============================================================================
# STAGE 2: PROMOTE RIGID EDGE TO CHAIN
# =============================================================================
# invivo_ref_red --[T_rigid]--> ex_vivo_block_red
#   becomes
# ex_vivo_block_red --[Identity]--> ex_vivo_rigid (ref node) --[T_rigid.invert()]--> invivo_ref_red
#
# Ref node on ex-vivo side so forward nonlinear direction ends at in-vivo
# (ground-truth frame). Idempotent: if ex_vivo_rigid already exists, no-op.

if 'ex_vivo_rigid' in g.nodes:
    print("Chain already exists (ex_vivo_rigid present). Skipping promote.")
else:
    assert 'invivo_ref_red' in g.nodes, "invivo_ref_red node missing"
    assert 'ex_vivo_block_red' in g.nodes, "ex_vivo_block_red node missing"
    assert 'ex_vivo_block_red' in g.edges.get('invivo_ref_red', {}), \
        "No rigid edge invivo_ref_red -> ex_vivo_block_red to promote"

    T_rigid = g.edges['invivo_ref_red']['ex_vivo_block_red']   # invivo -> ex_vivo

    allowed = (ca.TranslateRotate, ca.TranslateRotateRescale, ca.AffineTransform)
    assert isinstance(T_rigid, allowed), (
        f"Existing edge is {type(T_rigid).__name__}, not a rigid/affine. "
        "Refusing to promote — would put a non-rigid transform on the rigid leg. "
        "Re-fit rigid first (Mode D)."
    )

    n_nodes_before = len(list(g.nodes))
    n_edges_before = sum(len(g.edges[u]) for u in g.edges)

    g.remove_edge('invivo_ref_red', 'ex_vivo_block_red')       # tears both directions
    g.add_node('ex_vivo_rigid', image='invivo_ref_red')   # ref: fixed base shows in-vivo data
    g.add_edge('ex_vivo_block_red', 'ex_vivo_rigid', ca.Identity())       # nonlin placeholder
    g.add_edge('ex_vivo_rigid', 'invivo_ref_red', T_rigid.invert())       # rigid, fwd = ex -> in
    g.save()

    n_nodes_after = len(list(g.nodes))
    n_edges_after = sum(len(g.edges[u]) for u in g.edges)
    print(f"Promoted to chain.")
    print(f"  Nodes: {n_nodes_before} -> {n_nodes_after}")
    print(f"  Edges: {n_edges_before} -> {n_edges_after}")
    print(f"  Nonlin leg: ex_vivo_block_red -> ex_vivo_rigid  (Identity, ready for Stage 3)")
    print(f"  Rigid leg : ex_vivo_rigid -> invivo_ref_red  ({type(T_rigid.invert()).__name__})")
    print()
    print("WARNING: do NOT re-run Stage 1 (Mode D) on invivo_ref_red -> ex_vivo_block_red.")
    print("To re-fit rigid: Stage 4 (demote) -> Mode D -> Stage 2 (promote).")


---

## 9c. Stage 3 — Fit nonlinear (Triangulation) on `ex_vivo_block_red → ex_vivo_rigid`

Opens the GUI on the first leg of the chain. Press `V` for Triangulation,
add **≥4 non-coplanar** landmark pairs, then press `S` to save.

The save routes to edge `ex_vivo_block_red → ex_vivo_rigid` (replacing Identity)
because `movable_node='ex_vivo_block_red'` is explicit. Forward direction of this
edge (ex-vivo points → in-vivo frame) is the direction that matters for MNN:
it's the fast/exact traversal, since the triangulation is fitted *forward*.


In [ ]:
# =============================================================================
# STAGE 3: FIT NONLINEAR (TRIANGULATION) ON FIRST LEG OF CHAIN
# =============================================================================
# Layout: ex-vivo block (distorted) movable, in-vivo anatomical (ref, ground
# truth) fixed. ex_vivo_rigid resolves to invivo_ref_red image data via the ref
# node.
# Hotkeys: V = Triangulation (need >=4 non-coplanar points), S = save.

assert 'ex_vivo_rigid' in g.nodes, "Run Stage 2 (promote) first."

ca_gui.align_interactive(
    nodes_movable='ex_vivo_block_red',
    nodes_fixed='ex_vivo_rigid',
    graph=g,
)


---

## 9d. Stage 4 — (optional) Demote chain back to single rigid edge

Use only when you need to re-fit rigid. **Discards any nonlinear work** —
the landmark points were defined against the old rigid and are not
valid against a new rigid.

Workflow: Stage 4 (demote) → Mode D (re-fit rigid) → Stage 2 (promote) →
Stage 3 (re-fit nonlinear).


In [ ]:
# =============================================================================
# STAGE 4: DEMOTE CHAIN TO SINGLE RIGID EDGE (NONLINEAR DISCARDED)
# =============================================================================

if 'ex_vivo_rigid' not in g.nodes:
    print("No chain to demote (ex_vivo_rigid not present).")
else:
    T_rigid_exvivo_to_invivo = g.edges['ex_vivo_rigid']['invivo_ref_red']
    print(f"Capturing rigid transform: {type(T_rigid_exvivo_to_invivo).__name__}")

    g.remove_edge('ex_vivo_block_red', 'ex_vivo_rigid')
    g.remove_edge('ex_vivo_rigid', 'invivo_ref_red')
    g.remove_node('ex_vivo_rigid')

    # Restore the original direction invivo_ref_red -> ex_vivo_block_red so Mode D can
    # pick it up as the starting transform for a re-fit.
    g.add_edge('invivo_ref_red', 'ex_vivo_block_red', T_rigid_exvivo_to_invivo.invert())
    g.save()

    print("Demoted. Single rigid edge invivo_ref_red -> ex_vivo_block_red restored.")
    print("Nonlinear refinement (if any) was discarded.")


---

## 9e. Stage 5 — Query transforms for validation

Three queries expose rigid-only vs composed transforms for downstream
MNN validation (see `project_alignment_validation_mnn.md`).


In [ ]:
# =============================================================================
# STAGE 5: QUERY TRANSFORMS FOR VALIDATION
# =============================================================================
#   Rigid only    : g.get_transform('invivo_ref_red', 'ex_vivo_rigid')
#   Composed      : g.get_transform('invivo_ref_red', 'ex_vivo_block_red')
#   Nonlinear leg : g.get_transform('ex_vivo_block_red', 'ex_vivo_rigid')
# Before promote (chain absent), only the composed query works.
#
# For MNN validation on N>1000 centroids, prefer the forward direction
# g.get_transform('ex_vivo_block_red', 'invivo_ref_red') — forward nonlinear
# traversal is exact and fast, vs. backward which is numerical inversion
# capped at 1000 points (castalign/base.py:358).

for label, src, dst in [
    ("Rigid only    ", 'invivo_ref_red',    'ex_vivo_rigid'),
    ("Composed      ", 'invivo_ref_red',    'ex_vivo_block_red'),
    ("Nonlinear leg ", 'ex_vivo_block_red', 'ex_vivo_rigid'),
]:
    try:
        T = g.get_transform(src, dst)
        print(f"{label}: {type(T).__name__}")
    except Exception as e:
        print(f"{label}: unavailable ({e})")

print(f"\nNodes: {list(g.nodes)}")
print(f"Edges: {[(u, v) for u in g.edges for v in g.edges[u]]}")


---

## 9f. Stage 6 — Compare rigid-only vs rigid+nonlinear in two napari windows

Uses both halves of the chain independently, without mutating the graph:
- **Rigid only**: `g.edges['ex_vivo_rigid']['invivo_ref_red']` (single edge, bypasses the nonlinear leg).
- **Rigid + nonlinear**: `g.get_transform('ex_vivo_block_red', 'invivo_ref_red')` (BFS auto-composes both legs).

Warps ex-vivo into in-vivo frame under each transform and opens **two separate napari viewers** so you can screenshot each and compare placement / residuals at the same Z slice.

Requires Stage 3 (cell 9c) to have been completed — if the nonlinear leg is still Identity, the two viewers will show identical images and the cell refuses to run.

In [ ]:
# =============================================================================
# STAGE 6: COMPARE RIGID-ONLY vs RIGID+NONLINEAR (TWO NAPARI VIEWERS)
# =============================================================================
# Non-destructive: only reads the graph. Chain is preserved.

import napari

# --- Guards -----------------------------------------------------------------
assert 'ex_vivo_rigid' in g.nodes, (
    "Chain not present — run Stage 2 (cell 9b, promote) first."
)
T_rigid = g.edges['ex_vivo_rigid']['invivo_ref_red']       # 1-hop rigid
T_full  = g.get_transform('ex_vivo_block_red', 'invivo_ref_red')  # 2-hop composed

T_nonlin = g.edges['ex_vivo_block_red']['ex_vivo_rigid']
if isinstance(T_nonlin, ca.Identity):
    raise RuntimeError(
        "Nonlinear leg is still Identity — fit it via Stage 3 (cell 9c) first, "
        "otherwise the two viewers will be identical."
    )

# --- Images (shared between viewers) ---------------------------------------
fixed_img   = np.asarray(g.get_image('invivo_ref_red'))      # ground truth, fixed
movable_img = np.asarray(g.get_image('ex_vivo_block_red'))   # raw ex-vivo

# --- Warp under each transform --------------------------------------------
r_rigid, warped_rigid = compute_alignment_quality(fixed_img, movable_img, T_rigid)
r_full,  warped_full  = compute_alignment_quality(fixed_img, movable_img, T_full)

print(f"Rigid only          : {type(T_rigid).__name__:30s}  Pearson r = {r_rigid:.4f}")
print(f"Rigid + nonlinear   : {type(T_full).__name__:30s}  Pearson r = {r_full:.4f}")
print(f"Delta (full - rigid): {r_full - r_rigid:+.4f}")
print()

# --- Viewer 1: rigid only --------------------------------------------------
v_rigid = napari.Viewer(title="Rigid only")
v_rigid.add_image(
    fixed_img, name="in-vivo (fixed)",
    colormap="gray", blending="additive", opacity=0.7,
)
v_rigid.add_image(
    warped_rigid, name="ex-vivo warped (rigid)",
    colormap="green", blending="additive", opacity=0.5,
)

# --- Viewer 2: rigid + nonlinear ------------------------------------------
v_full = napari.Viewer(title="Rigid + nonlinear")
v_full.add_image(
    fixed_img, name="in-vivo (fixed)",
    colormap="gray", blending="additive", opacity=0.7,
)
v_full.add_image(
    warped_full, name="ex-vivo warped (rigid+nonlin)",
    colormap="green", blending="additive", opacity=0.5,
)

print("Two viewers open. Scroll both to the same Z slice, then screenshot each.")


---

## 10. Post-Alignment Verification

Applies the current transform, computes Pearson correlation, and opens a napari viewer with three layers:
- **Fixed** (gray) — the reference image
- **Warped** (green) — the transformed movable image
- **Original** (red, hidden) — the untransformed movable for comparison

Set `verify_target` to choose which alignment to verify:
- `block_node` — direct slice → block edge
- `invivo_node` — full two-hop path (slice → block → invivo, composed automatically)

In [ ]:
# =============================================================================
# POST-ALIGNMENT VERIFICATION
# =============================================================================

# --- Choose verification target ---
slice_name = get_selected_slice()

# Option 1: Verify slice -> block (direct edge)
verify_target = block_node
# Option 2: Verify slice -> invivo (full two-hop, composed automatically)
# verify_target = invivo_node

verify_fixed = g.get_image(verify_target)
verify_movable = g.get_image(slice_name)
verify_transform = g.get_transform(slice_name, verify_target)

print(f"Verifying alignment: {slice_name} -> {verify_target}")
print(f"Transform: {type(verify_transform).__name__}")
print()

# Compute quality
r, warped_arr = compute_alignment_quality(verify_fixed, verify_movable, verify_transform)
print(f"Pearson correlation (overlapping region): r = {r:.4f}")
print()

# Open napari viewer
import napari

viewer = napari.Viewer(title=f"Verification: {slice_name} -> {verify_target}")
viewer.add_image(np.asarray(verify_fixed), name="fixed", colormap="gray", blending="additive", opacity=0.7)
viewer.add_image(warped_arr, name="warped", colormap="green", blending="additive", opacity=0.5)
viewer.add_image(np.asarray(verify_movable), name="original (hidden)", colormap="red", blending="additive", opacity=0.3, visible=False)

print("Napari viewer opened:")
print("  gray  = fixed reference")
print("  green = warped movable")
print("  red   = original movable (hidden, toggle in layer list)")

---

## 11. Nonlinear Refinement: Triangulation2D

**Use after:** Initial affine alignment to block (Mode A, B, or C)

**What it does:** Adds nonlinear warping to correct edge distortions in the slice → block alignment.

**How it works:**
- Uses padding (required for point-based Triangulation2D)
- Starts from existing transform, adds nonlinear refinement

**How to use:**
1. Select an already-aligned slice
2. Run this cell
3. Press `N` (projected) or `V` (3D) when prompted
4. Click corresponding points on edges that need warping
5. Press 'q' to quit and save

In [ ]:
# =============================================================================
# NONLINEAR REFINEMENT: TRIANGULATION (requires padding)
# Same role layout as Mode A: movable = padded slice, fixed = block.
# =============================================================================

slice_name = get_selected_slice()

if slice_name not in aligned_slices:
    print(f"ERROR: {slice_name} is not aligned yet!")
    print("Run one of the alignment modes first.")
else:
    print(f"Refining: {slice_name}")
    print(f"Mode: Nonlinear refinement (N or V transforms)")
    print()

    # Existing transform in movable->fixed direction (slice -> block)
    existing_transform = g.get_transform(slice_name, block_node)
    print(f"Existing transform: {type(existing_transform).__name__}")

    PAD_Z = 25
    movable_img = load_and_pad_slice(slice_name, pad_z=PAD_Z)
    fixed_img   = g.get_image(block_node)

    print(f"Movable (padded {slice_name}): {movable_img.shape}")
    print(f"Fixed   ({block_node}):        {fixed_img.shape}")

    print("\n" + "="*60)
    print("LAUNCHING GUI FOR REFINEMENT")
    print("="*60)
    print("Starting from existing transform")
    print("Use 'N' for projected triangulation or 'V' for 3D triangulation")
    print("Press 'q' to quit when done")
    print()

    refined_transform = ca_gui.align_interactive(
        nodes_movable=movable_img,
        nodes_fixed=fixed_img,
        transform=existing_transform,
        graph=g,
    )

    save_alignment(slice_name, refined_transform)

    del movable_img
    gc.collect()


---

## 12. Warped Image Export

Save a warped volume as a multi-page TIFF. Set `export_target` to choose the output coordinate space:
- `block_node` — block space (direct transform)
- `invivo_node` — in-vivo space (two-hop composed automatically)

In [ ]:
# =============================================================================
# WARPED IMAGE EXPORT
# =============================================================================

# Which slice to export
slice_name = get_selected_slice()

# Choose output coordinate space
export_target = invivo_node  # or block_node for block space
export_target_img = g.get_image(export_target)

# Reuse warped_arr from verification cell if it exists
try:
    assert warped_arr is not None
    print(f"Reusing warped array from verification cell")
except (NameError, AssertionError):
    # Recompute
    print(f"Computing warped image for: {slice_name} -> {export_target}")
    t = g.get_transform(slice_name, export_target)
    slice_img = g.get_image(slice_name)
    _, warped_arr = compute_alignment_quality(export_target_img, slice_img, t)

# Export
output_path = WARPED_OUTPUT_DIR / f"{slice_name}_warped.tiff"
export_warped_image(warped_arr, output_path, dtype=np.float32)

---

## 13. Graph Reload Verification

Validates that the graph .db file correctly persists the slice → block transform through SQLite serialization. Saves the graph, reloads it, re-applies the transform, and checks that the output matches.

In [ ]:
# =============================================================================
# GRAPH RELOAD VERIFICATION
# =============================================================================

slice_name = get_selected_slice()

if slice_name not in aligned_slices:
    print(f"ERROR: {slice_name} is not aligned yet!")
else:
    # Step 1: Get current transform output
    print(f"Verifying graph persistence for: {slice_name} -> {block_node}")
    t_original = g.get_transform(slice_name, block_node)
    slice_img = g.get_image(slice_name)
    block_img = g.get_image(block_node)
    
    warped_original = np.asarray(t_original.transform_image(
        slice_img, output_size=list(block_img.shape)
    ))
    
    # Step 2: Save graph
    g.save()
    print(f"  Graph saved to: {GRAPH_PATH}")
    
    # Step 3: Reload graph
    g_check = ca.Graph.load(str(GRAPH_PATH))
    t_reloaded = g_check.get_transform(slice_name, block_node)
    slice_img_reload = g_check.get_image(slice_name)
    
    warped_reloaded = np.asarray(t_reloaded.transform_image(
        slice_img_reload, output_size=list(block_img.shape)
    ))
    
    # Step 4: Compare
    max_diff = np.max(np.abs(warped_original.astype(np.float64) - warped_reloaded.astype(np.float64)))
    mean_diff = np.mean(np.abs(warped_original.astype(np.float64) - warped_reloaded.astype(np.float64)))
    
    print(f"\n  Original transform:  {type(t_original).__name__}")
    print(f"  Reloaded transform:  {type(t_reloaded).__name__}")
    print(f"  Max absolute diff:   {max_diff:.6e}")
    print(f"  Mean absolute diff:  {mean_diff:.6e}")
    
    if max_diff < 1e-3:
        print(f"\n  PASS: Graph persistence verified")
    else:
        print(f"\n  WARNING: Max diff {max_diff:.6e} exceeds threshold 1e-3")
        print(f"  This may indicate SQLite serialization issues (check Samba mount)")
    
    del g_check, warped_original, warped_reloaded
    gc.collect()

---

## 14. GraphViewer: See All Aligned Slices

Visualize all aligned slices together in the in-vivo coordinate space.

In [ ]:
# =============================================================================
# GRAPHVIEWER: VISUALIZE ALL NODES ALIGNED TO INVIVO
# =============================================================================
# Includes subslices aligned via block (slice->block->invivo) AND the block
# itself if aligned directly to invivo (Mode D).

print("=" * 60)
print("GRAPHVIEWER")
print("=" * 60)

# Collect every non-invivo node that has a transform path to invivo_node.
aligned_to_invivo = []
for n in g.nodes:
    if n == invivo_node:
        continue
    try:
        g.get_transform(n, invivo_node)
        aligned_to_invivo.append(n)
    except Exception:
        pass

print(f"Nodes aligned to {invivo_node}: {len(aligned_to_invivo)}")
for n in aligned_to_invivo:
    print(f"  - {n}")
print()

if len(aligned_to_invivo) == 0:
    print("Nothing aligned to invivo yet. Run an alignment mode first.")
else:
    colormaps = ["red", "green", "blue", "cyan", "magenta", "yellow"]

    if GraphViewer is None:
        print("GraphViewer not available; using manual napari viewer.\n")
        import napari
        v = napari.Viewer()
        v.add_image(g.get_image(invivo_node), name=invivo_node, colormap="gray", opacity=0.7)

        for i, name in enumerate(sorted(aligned_to_invivo)):
            try:
                img = g.get_image(name)
                transform = g.get_transform(name, invivo_node)
                transformed = transform.transform_image(img)
                origin = transformed.origin if hasattr(transformed, "origin") else [0, 0, 0]
                cmap = colormaps[i % len(colormaps)]
                v.add_image(transformed, name=name, colormap=cmap, opacity=0.5, translate=origin)
                print(f"  Added: {name} ({cmap})")
            except Exception as e:
                print(f"  Failed: {name} - {e}")

        print("\n\u2713 Viewer opened")
        napari.run()

    else:
        print(f"Opening GraphViewer in {invivo_node} space...\n")
        v = GraphViewer(g, space=invivo_node)
        v.add_image(invivo_node, colormap="gray", opacity=0.7)

        for i, name in enumerate(sorted(aligned_to_invivo)):
            try:
                cmap = colormaps[i % len(colormaps)]
                v.add_image(name, colormap=cmap, opacity=0.5)
                print(f"  Added: {name} ({cmap})")
            except Exception as e:
                print(f"  Failed: {name} - {e}")

        print("\n\u2713 GraphViewer opened")


---

## 15. Alignment Status Summary

Quick overview of all slices and their alignment status, including transform types and Mode D alignments.

In [ ]:
# =============================================================================
# ALIGNMENT STATUS SUMMARY
# =============================================================================

# Refresh alignment status from graph
aligned_slices = []
unaligned_slices = []

for s in slice_nodes:
    if s in g.edges and block_node in g.edges[s]:
        aligned_slices.append(s)
    else:
        unaligned_slices.append(s)

block_to_invivo_aligned = (
    block_node in g.edges and invivo_node in g.edges.get(block_node, {})
)

print("="*60)
print("ALIGNMENT STATUS")
print("="*60)

if len(slice_nodes) > 0:
    print(f"Total slices: {len(slice_nodes)}")
    print(f"Aligned to block: {len(aligned_slices)} ({100*len(aligned_slices)/len(slice_nodes):.1f}%)")
    print(f"Remaining: {len(unaligned_slices)}")
    print()

    # Progress bar
    progress = len(aligned_slices) / len(slice_nodes)
    bar_width = 40
    filled = int(bar_width * progress)
    bar = "=" * filled + "-" * (bar_width - filled)
    print(f"Progress: [{bar}] {100*progress:.1f}%")
    print()
else:
    print("Subslices in graph: 0 (none yet -- graph has block+invivo only)")
    print()

# Block -> InVivo status
if block_to_invivo_aligned:
    t_bi = g.get_transform(block_node, invivo_node)
    print(f"Block -> InVivo: ALIGNED ({type(t_bi).__name__})")
else:
    print(f"Block -> InVivo: NOT ALIGNED (run Mode D)")
print()

# Generalized "aligned to invivo" roll-up (traverses graph; covers any chain)
aligned_to_invivo = []
for n in g.nodes:
    if n == invivo_node:
        continue
    try:
        t = g.get_transform(n, invivo_node)
        aligned_to_invivo.append((n, type(t).__name__))
    except Exception:
        pass

print("Aligned to in-vivo:")
if aligned_to_invivo:
    for n, tname in sorted(aligned_to_invivo):
        print(f"  + {n}: {tname}")
else:
    print("  (none)")
print()

# List aligned slices with transform type (only meaningful when subslices exist)
if slice_nodes and aligned_slices:
    print("Aligned slices (-> block):")
    transform_counts = {}
    for s in sorted(aligned_slices):
        t = g.get_transform(s, block_node)
        tname = type(t).__name__
        transform_counts[tname] = transform_counts.get(tname, 0) + 1
        print(f"  + {s}: {tname}")
    print()
    print("Transform summary:")
    for tname, count in sorted(transform_counts.items()):
        print(f"  {tname}: {count} slices")
    print()

# List next few unaligned (only when subslices exist)
if slice_nodes and unaligned_slices:
    print("Next unaligned slices:")
    for s in sorted(unaligned_slices)[:5]:
        print(f"  o {s}")
    if len(unaligned_slices) > 5:
        print(f"  ... and {len(unaligned_slices) - 5} more")
    print()

# Update dropdown
slice_dropdown.options = get_slice_options()

print("="*60)


---

## 16. Graph Structure Visualization

View the graph structure as a node-edge diagram.

In [ ]:
# =============================================================================
# GRAPH STRUCTURE VISUALIZATION
# =============================================================================

print(f"Graph: {g.name}")
print(f"Nodes: {len(g.nodes)}")
print(f"Edges: {sum(len(e) for e in g.edges.values()) // 2}")
print()

# Visualize (uses SVG via patch)
g.visualise()

---

## 17. Advanced: `alignment_gui()` with Crop & References

Power-user cell showing how to call `alignment_gui()` directly instead of `align_interactive()`.

**Trade-off:** `alignment_gui()` handles a single transform at a time (no composition loop like `align_interactive()`), but supports two extra parameters:
- **`crop=True`** — only renders the region of the movable image that overlaps the fixed image, saving memory
- **`references`** — list of additional images to display as alignment aids

**Note:** `crop` does NOT work with `align_interactive()` — it's only available through `alignment_gui()`.

In [ ]:
# =============================================================================
# ADVANCED: alignment_gui() with crop and references
# =============================================================================
# alignment_gui() is the lower-level function that align_interactive() calls
# internally. Opens a single napari window for one transform at a time.
#
# Advantages:
#   - crop=True renders only the overlap region (lower RAM)
#   - references=[] shows additional images as guides
# Disadvantages:
#   - No composition loop -- one transform per call
#   - No save-to-graph shortcuts -- you manage saving yourself
#
# Role layout matches Mode A: movable = padded slice, fixed = block.
# =============================================================================

slice_name = get_selected_slice()

PAD_Z = 25
movable_img = load_and_pad_slice(slice_name, pad_z=PAD_Z)
fixed_img   = g.get_image(block_node)

# Starting transform (extend with point-based rigid)
prev_transform = get_previous_transform(slice_name)
if prev_transform is not None:
    initial_transform = prev_transform + ca.TranslateRotate
else:
    initial_transform = get_initial_transform_slice_to_block() + ca.TranslateRotate

# Optional reference images as guides: list of (image, transform) tuples
references = []

print(f"Movable (padded {slice_name}): {movable_img.shape}")
print(f"Fixed   ({block_node}):        {fixed_img.shape}")
print("Launching alignment_gui() with crop=True (overlap-only rendering)")
print()

transform = ca_gui.alignment_gui(
    movable_image=movable_img,
    base_image=fixed_img,
    transform=initial_transform,
    crop=True,
    references=references,
)

save_alignment(slice_name, transform)

del movable_img
gc.collect()


---

## 18. Advanced: Node-Name Mode

Instead of passing raw arrays to `align_interactive()`, you can pass **node names** (strings) and let CASTalign resolve images from the graph. This is simpler but doesn't work with padding (Modes A/C) since the padded array isn't stored as a graph node.

**Use when:** Both images are already stored in the graph as nodes (e.g., Mode D, or unpadded Mode B).

In [ ]:
# =============================================================================
# ADVANCED: NODE-NAME MODE
# =============================================================================
# When both images are stored as graph nodes, pass their names (strings)
# and castalign calls g.get_image() internally.
#
# Works when:
#   - Both nodes exist in the graph
#   - You don't need padding (graph stores raw image, not padded)
#   - graph=g is passed
#
# Does NOT work for Modes A/C (they pad the slice before alignment).
#
# Role layout: movable = slice node, fixed = block node.
# =============================================================================

slice_name = get_selected_slice()

movable_node = slice_name
fixed_node   = block_node

print(f"Node-name alignment: movable={movable_node}, fixed={fixed_node}")
print(f"(Images will be loaded from graph automatically)")
print()

prev_transform = get_previous_transform(slice_name)
initial_transform = prev_transform if prev_transform is not None else get_initial_transform_slice_to_block()

transform = ca_gui.align_interactive(
    nodes_movable=movable_node,
    nodes_fixed=fixed_node,
    transform=initial_transform,
    graph=g,
)

save_alignment(slice_name, transform)


---

## 19. Troubleshooting & Tips

### Two-Hop Graph Structure
- Slices align to the ex-vivo block, and the block aligns to in-vivo.
- `g.get_transform(slice, invivo_ref_red)` automatically composes both transforms via BFS.
- The block → invivo transform (Mode D) only needs to be done once.
- If you see "Path not found" errors, check that both edges exist (slice→block AND block→invivo).

### SQLite over Samba/SMB
- Graph `.db` files are SQLite databases. Writing over network mounts can cause "database is locked" or corruption errors.
- **Fix:** Copy the `.db` file locally, do alignment, copy back. Or run alignment directly on the server via SSH + X11 forwarding.
- Use the Graph Reload Verification cell (Cell 13) to confirm persistence after saving.

### Memory Management
- The block and in-vivo stacks (~200 MB each) stay in RAM once loaded by the graph. Padded slices add ~100 MB each.
- After alignment, `del padded; gc.collect()` frees the padded array.
- If napari becomes sluggish, restart the kernel and re-run from Cell 1.
- `crop=True` in `alignment_gui()` (Cell 17) reduces memory by only rendering the overlap region.

### Padding Rationale
- Point-based transforms (T, R, P, N, V) need at least a few z-slices to fit 3D correspondences.
- A 1-voxel slice has no z-extent, so the fitter can't determine z-position.
- We pad to `2*PAD_Z+1` slices (default 51) with the original at center (z=25).
- Padding value = 90% of slice mean to avoid confusing the fitter with zeros or max values.

### Label Images & `image_is_label`
- CASTalign auto-detects label images (integer dtype, few unique values) and uses nearest-neighbor interpolation.
- Cell masks and other continuous-valued images can be misdetected as labels.
- We disable this globally with `ca_utils.image_is_label = lambda img: False` in the imports cell.
- To re-enable for specific images, pass `labels=True` to `transform_image()`.

### Transform Composition & Inversion
- Transforms compose left-to-right: `t1 + t2` means "apply t1 first, then t2".
- The graph stores directed edges. `g.get_transform(A, B)` returns the A->B direction.
- CASTalign automatically inverts transforms when traversing edges in reverse.
- For two-hop: `g.get_transform(slice, invivo)` finds path slice→block→invivo and composes both transforms.

### Common Issues
- **"Point-based transforms don't fit"** — Make sure you have at least 3 point pairs (4+ recommended). Check that points are on the correct layers (base vs movable).
- **"Transform looks wrong after loading"** — Run the Graph Reload Verification cell. If it fails, the SQLite file may be corrupted (Samba issue).
- **"GUI shows blank/black"** — The images may have very different intensity ranges. Try normalizing before alignment, or adjust napari contrast limits manually.
- **"Path not found between nodes"** — The two-hop path requires both slice→block and block→invivo edges. Run Mode D if the block→invivo edge is missing.

---

## 20. MNN Alignment Validation

Quantitative alignment-quality metric built from Cellpose centroids + the fitted rigid transform. For each cell in the in-vivo volume, apply the `invivo_ref_red -> ex_vivo_block_red` transform, then mutual-nearest-neighbor (MNN) match against the ex-vivo centroids. Report the distance distribution in voxel and um.

**Prereqs:** Cellpose 3D has been run on both `BLOCK_STACK_PATH` and `INVIVO_PATH`. The `_seg.npy` files live in `<INVIVO_PATH parent>/cellpose/`.

Logic lives in `alignment/validate_mnn.py`; the cells below are thin wrappers around its functions so you can run and inspect step-by-step. Standalone CLI equivalent: `python alignment/validate_mnn.py`.


In [ ]:
# =============================================================================
# MNN VALIDATION - EXTRACT CENTROIDS
# =============================================================================
# Prereqs: cellpose _seg.npy files for both volumes, by default in
# <INVIVO_PATH parent>/cellpose/<stem>_seg.npy
# =============================================================================

# Make alignment/ importable (notebook cwd is already alignment/)
_ALIGN_DIR = Path.cwd()
if str(_ALIGN_DIR) not in sys.path:
    sys.path.insert(0, str(_ALIGN_DIR))

from validate_mnn import (
    extract_centroids,
    apply_transform_to_centroids,
    mnn_match,
    summarize,
    print_summary,
    HUANG_UM_PER_VOXEL,
)

from local_config import INVIVO_PATH, BLOCK_STACK_PATH
invivo_path = Path(INVIVO_PATH)
exvivo_path = Path(BLOCK_STACK_PATH)
out_dir = invivo_path.parent / "cellpose"

# Override paths here if your cellpose _seg.npy files live elsewhere
exvivo_seg = out_dir / f"{exvivo_path.stem}_seg.npy"
invivo_seg = out_dir / f"{invivo_path.stem}_seg.npy"

print(f"Ex-vivo seg: {exvivo_seg}  ({'OK' if exvivo_seg.exists() else 'MISSING'})")
print(f"In-vivo seg: {invivo_seg}  ({'OK' if invivo_seg.exists() else 'MISSING'})")
print()

d_ex = extract_centroids(exvivo_seg)
d_in = extract_centroids(invivo_seg)

ex_size_med = float(np.median(d_ex['sizes'])) if d_ex['sizes'].size else float('nan')
in_size_med = float(np.median(d_in['sizes'])) if d_in['sizes'].size else float('nan')
print(f"Ex-vivo: {len(d_ex['centroids']):>6d} cells, median size {ex_size_med:.0f} voxels")
print(f"In-vivo: {len(d_in['centroids']):>6d} cells, median size {in_size_med:.0f} voxels")


In [ ]:
# =============================================================================
# MNN VALIDATION - APPLY RIGID TRANSFORM
# =============================================================================
# Queries the in-vivo -> ex-vivo-block edge (auto-composed for multi-hop paths).
# Centroids move from in-vivo voxel frame into ex-vivo voxel frame.
# =============================================================================

transform = g.get_transform(invivo_node, block_node)
print(f"Transform: {type(transform).__name__}")

in_in_ex_frame = apply_transform_to_centroids(transform, d_in['centroids'])

def _bbox_line(pts, label):
    if pts.shape[0] == 0:
        return f"{label}: (empty)"
    mn, mx = pts.min(axis=0), pts.max(axis=0)
    return (f"{label}: z {mn[0]:7.1f}-{mx[0]:7.1f}  "
            f"y {mn[1]:7.1f}-{mx[1]:7.1f}  x {mn[2]:7.1f}-{mx[2]:7.1f}")

print()
print(_bbox_line(d_ex['centroids'],  "ex-vivo             "))
print(_bbox_line(d_in['centroids'],  "in-vivo (raw)       "))
print(_bbox_line(in_in_ex_frame,     "in-vivo (ex-frame)  "))


In [ ]:
# =============================================================================
# MNN VALIDATION - MUTUAL NEAREST NEIGHBOR MATCH + SUMMARY
# =============================================================================
# KDTree bidirectional NN in voxel space. Distances reported in voxel and um
# (per-axis, since Z pitch is ~1.8x XY pitch).
# =============================================================================

mnn = mnn_match(in_in_ex_frame, d_ex['centroids'])
summary = summarize(
    mnn,
    n_movable=len(d_in['centroids']),
    n_fixed=len(d_ex['centroids']),
)
print_summary(summary)


In [ ]:
# =============================================================================
# MNN VALIDATION - NAPARI OVERLAY
# =============================================================================
# Plots both centroid sets on the ex-vivo volume in the ex-vivo frame.
# Matched points -> green, unmatched -> grey. Visual QC of MNN matches.
# =============================================================================
import napari

ex_colors = np.tile([0.5, 0.5, 0.5, 0.6], (len(d_ex['centroids']), 1))
in_colors = np.tile([0.5, 0.5, 0.5, 0.6], (len(in_in_ex_frame), 1))
if mnn['fixed_idx'].size:
    ex_colors[mnn['fixed_idx']] = [0.0, 1.0, 0.0, 1.0]
if mnn['movable_idx'].size:
    in_colors[mnn['movable_idx']] = [0.0, 1.0, 0.0, 1.0]

viewer = napari.Viewer(title="MNN validation (ex-vivo frame)")
viewer.add_image(
    np.asarray(g.get_image(block_node)),
    name="ex_vivo_block_red", colormap="gray", blending="additive", opacity=0.7,
)
viewer.add_points(
    d_ex['centroids'], name="ex-vivo centroids",
    face_color=ex_colors, size=4, symbol='disc',
)
viewer.add_points(
    in_in_ex_frame, name="in-vivo centroids (transformed)",
    face_color=in_colors, size=4, symbol='cross',
)

print("Napari viewer opened:")
print("  disc  = ex-vivo centroids")
print("  cross = in-vivo centroids transformed into ex-vivo frame")
print("  green = MNN-matched pair, grey = unmatched")


In [ ]:
# =============================================================================
# MNN VALIDATION - DISTANCE HISTOGRAM
# =============================================================================
# Saves mnn_histogram.png next to the other cellpose-derived outputs.
# =============================================================================
import matplotlib.pyplot as plt

d_um = np.linalg.norm(mnn['distances_um_per_axis'], axis=1)
d_vox = mnn['distances_voxel']

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, data, unit in [(axes[0], d_vox, 'voxel'), (axes[1], d_um, 'um')]:
    if data.size == 0:
        ax.text(0.5, 0.5, "no MNN matches", ha='center', va='center', transform=ax.transAxes)
        ax.set_xlabel(f"MNN distance ({unit})")
        continue
    ax.hist(data, bins=40, color='steelblue', alpha=0.85)
    med = float(np.median(data))
    p95 = float(np.percentile(data, 95))
    ax.axvline(med, color='red',    linestyle='--', label=f"median {med:.2f}")
    ax.axvline(p95, color='orange', linestyle='--', label=f"p95 {p95:.2f}")
    ax.set_xlabel(f"MNN distance ({unit})")
    ax.set_ylabel("count")
    ax.legend()

plt.tight_layout()
out_hist = out_dir / "mnn_histogram.png"
out_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(out_hist, dpi=120)
print(f"Saved: {out_hist}")
plt.show()
